In [ ]:
from sklearn.metrics import f1_score

def evaluate(model, loader, device):

    model.eval()

    total_loss = 0

    punct_preds = []
    punct_true = []

    cap_preds = []
    cap_true = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            punct_labels = batch["punct_labels"].to(device)
            cap_labels = batch["cap_labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                punct_labels=punct_labels,
                cap_labels=cap_labels
            )

            loss = outputs["loss"]

            punct_logits = outputs["punct_logits"]
            cap_logits = outputs["cap_logits"]

            total_loss += loss.item()

            punct_predictions = torch.argmax(punct_logits, dim=-1)
            cap_predictions = torch.argmax(cap_logits, dim=-1)

            punct_preds.extend(punct_predictions.cpu().numpy().flatten())
            punct_true.extend(punct_labels.cpu().numpy().flatten())

            cap_preds.extend(cap_predictions.cpu().numpy().flatten())
            cap_true.extend(cap_labels.cpu().numpy().flatten())

    # удаляем -100
    def filter_labels(preds, true):

        f_preds = []
        f_true = []

        for p, t in zip(preds, true):

            if t != -100:
                f_preds.append(p)
                f_true.append(t)

        return f_preds, f_true


    punct_preds, punct_true = filter_labels(punct_preds, punct_true)
    cap_preds, cap_true = filter_labels(cap_preds, cap_true)

    punct_f1 = f1_score(punct_true, punct_preds, average="macro")
    cap_f1 = f1_score(cap_true, cap_preds, average="macro")

    return total_loss / len(loader), punct_f1, cap_f1

In [ ]:
from torch.optim import AdamW
from sklearn.metrics import f1_score
import copy

optimizer = AdamW(model.parameters(), lr=3e-5)

best_val_loss = float("inf")
best_weights = None

patience = 3
epochs_without_improvement = 0
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    punct_preds = []
    punct_true = []

    cap_preds = []
    cap_true = []

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        punct_labels = batch["punct_labels"].to(device)
        cap_labels = batch["cap_labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            punct_labels=punct_labels,
            cap_labels=cap_labels
        )

        loss = outputs["loss"]
        punct_logits = outputs["punct_logits"]
        cap_logits = outputs["cap_logits"]

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        punct_predictions = torch.argmax(punct_logits, dim=-1)
        cap_predictions = torch.argmax(cap_logits, dim=-1)

        punct_preds.extend(punct_predictions.cpu().numpy().flatten())
        punct_true.extend(punct_labels.cpu().numpy().flatten())

        cap_preds.extend(cap_predictions.cpu().numpy().flatten())
        cap_true.extend(cap_labels.cpu().numpy().flatten())

    # ---- фильтрация -100 ----
    def filter_labels(preds, true):

        f_preds = []
        f_true = []

        for p, t in zip(preds, true):

            if t != -100:
                f_preds.append(p)
                f_true.append(t)

        return f_preds, f_true


    punct_preds, punct_true = filter_labels(punct_preds, punct_true)
    cap_preds, cap_true = filter_labels(cap_preds, cap_true)

    punct_f1 = f1_score(punct_true, punct_preds, average="macro")
    cap_f1 = f1_score(cap_true, cap_preds, average="macro")

    train_loss = total_loss / len(train_loader)

    # ---- validation ----
    val_loss, val_punct_f1, val_cap_f1 = evaluate(model, val_loader, device)

    print(f"\nEpoch {epoch}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Train punct F1: {punct_f1:.4f} | Train cap F1: {cap_f1:.4f}")
    print(f"Val   loss: {val_loss:.4f}")
    print(f"Val punct F1: {val_punct_f1:.4f} | Val cap F1: {val_cap_f1:.4f}")

    # ---- early stopping ----
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())

        epochs_without_improvement = 0
        print("New best model")

    else:

        epochs_without_improvement += 1
        print(f"No improvement ({epochs_without_improvement}/{patience})")

        if epochs_without_improvement >= patience:
            print("\nEarly stopping triggered")
            break


# восстановление лучших весов
model.load_state_dict(best_weights)
print("\nBest model restored")